# 🚀 Phase 8 Training Notebook (Anti-Overfitting)

## Key Fixes from Phase 7 Analysis:
- **MICRO tier model** - 0.65M params (was 11.98M MEDIUM - too large for 358 samples)
- **Lower learning rate** - 1e-5 for fine-tuning (was 1e-4)
- **Only 5 epochs** - Early stopping to prevent overfitting (was 15)
- **Fresh training** - Train from scratch with proper regularization
- **Increased dropout** - 0.3 audio/video dropout
- **Clean checkpoint dir** - Remove old mixed checkpoints

## Phase 7 Problems:
| Issue | Symptom |
|-------|---------|
| **Too many epochs** | Train F1=0.82, Val F1=0.08 (collapse) |
| **Model too large** | 11.98M params for ~286 samples |
| **LR too high** | Rapid overfitting after epoch 5-6 |

## Phase 8 Targets:
| Metric | Target |
|--------|--------|
| **Ensemble F1** | **0.68+** |
| **Recall** | **0.85+** |
| **Precision** | **0.55+** |

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone/Update Repository

In [ ]:
import os

if not os.path.exists('/content/phase2'):
    !git clone https://github.com/nithin12342/phase2.git /content/phase2
    print("✅ Repository cloned successfully!")
else:
    %cd /content/phase2
    !git fetch origin
    !git reset --hard origin/main
    %cd /content
    print("✅ Repository updated to latest!")

## Step 3: Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers h5py pandas scikit-learn tqdm --quiet
print("✅ Dependencies installed!")

## Step 4: Configure Paths & Clean Phase 8 Directory

In [ ]:
# ============================================
# Phase 8 Configuration - Anti-Overfitting
# ============================================

OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"
LABELS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv"

# Phase 8 output directory (clean start)
OUT_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase8"

# Use MICRO tier to prevent overfitting (0.65M vs 11.98M params)
TIER = "micro"

# Training hyperparameters
EPOCHS = 5  # Much fewer epochs to prevent overfitting
LR = 1e-5   # Lower LR for stability

import os
import glob

# Clean checkpoint directory for fresh start
!rm -rf {OUT_DIR}
!mkdir -p {OUT_DIR}
print(f"🧹 Cleaned and created: {OUT_DIR}")

# Verify paths
print(f"\n📁 OS_PATH exists: {os.path.exists(OS_PATH)}")
print(f"📁 DATA_DIR exists: {os.path.exists(DATA_DIR)}")
print(f"📁 LABELS exists: {os.path.exists(LABELS)}")
print(f"\n⚙️ Configuration:")
print(f"   Tier: {TIER} (smaller model to prevent overfitting)")
print(f"   Epochs: {EPOCHS} (fewer to prevent overfitting)")
print(f"   Learning Rate: {LR} (lower for stability)")

## Step 5: Train All 5 Folds (Fresh Start with MICRO Tier)

In [ ]:
# ============================================
# Phase 8 Training - Anti-Overfitting Strategy
# ============================================
# Key changes:
# 1. MICRO tier (0.65M params) instead of MEDIUM (11.98M)
# 2. Only 5 epochs instead of 15
# 3. Lower learning rate (1e-5)
# 4. Fresh training (no resume from Phase 6/7)

import time

total_start = time.time()

for fold in range(5):
    print(f"\n{'='*60}")
    print(f"🚀 PHASE 8 - FOLD {fold}/4 (Fresh Training with MICRO tier)")
    print(f"{'='*60}\n")

    fold_start = time.time()

    # Train from scratch with anti-overfitting settings
    !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/train.py \
        --data_dir {DATA_DIR} \
        --labels_csv {LABELS} \
        --output_dir {OUT_DIR} \
        --tier {TIER} \
        --epochs {EPOCHS} \
        --fold_idx {fold} \
        --lr {LR}

    fold_time = time.time() - fold_start
    print(f"\n⏱️ Fold {fold} completed in {fold_time/60:.1f} minutes")

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"✅ ALL 5 FOLDS COMPLETED!")
print(f"⏱️ Total training time: {total_time/60:.1f} minutes")
print(f"{'='*60}")

## Step 6: Verify Checkpoints

In [ ]:
import os
import glob

print("📦 Phase 8 Checkpoints:")
checkpoints = sorted(glob.glob(f"{OUT_DIR}/*_best.pt"))

if not checkpoints:
    print("   ⚠️ No checkpoints found! Training may have failed.")
else:
    for ckpt in checkpoints:
        size_mb = os.path.getsize(ckpt) / (1024*1024)
        print(f"   ✅ {os.path.basename(ckpt)} ({size_mb:.1f} MB)")
    print(f"\n✅ Total: {len(checkpoints)} checkpoints ready for ensemble!")

## Step 7: Run Ensemble Prediction

In [ ]:
print("🔮 Generating ensemble predictions...\n")

!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints {OUT_DIR} \
    --input {DATA_DIR} \
    --tier {TIER} \
    --output "/content/phase8_results.csv"

import os
if os.path.exists("/content/phase8_results.csv"):
    print("\n✅ Ensemble predictions saved to /content/phase8_results.csv")
else:
    print("\n❌ Ensemble prediction failed!")

## Step 8: Evaluate Final Results

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix

print("🏆 Calculating Final Phase 8 Results...")

# Load results and labels
results_df = pd.read_csv("/content/phase8_results.csv")
labels_df = pd.read_csv(LABELS)

# Merge on PID
results_df['pid'] = results_df['pid'].astype(str)
labels_df['Participant_ID'] = labels_df['Participant_ID'].astype(str)
merged = results_df.merge(labels_df, left_on='pid', right_on='Participant_ID', how='inner')

# Get predictions and true labels
y_pred = merged['prediction'].values
y_prob = merged['probability'].values

# Derive binary labels from PHQ8_Score
if 'PHQ8_Binary' in merged.columns:
    y_true = merged['PHQ8_Binary'].values
else:
    y_true = (merged['PHQ8_Score'] >= 10).astype(int).values
    print("💡 Derived binary labels from PHQ8_Score (threshold >= 10)")

# Calculate metrics at default threshold
threshold = 0.45
y_pred_thresh = (y_prob >= threshold).astype(int)

f1 = f1_score(y_true, y_pred_thresh)
precision = precision_score(y_true, y_pred_thresh)
recall = recall_score(y_true, y_pred_thresh)
accuracy = accuracy_score(y_true, y_pred_thresh)
cm = confusion_matrix(y_true, y_pred_thresh)

print(f"\n{'='*40}")
print(f"📊 PHASE 8 FINAL RESULTS (Threshold={threshold})")
print(f"{'='*40}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Total Samples: {len(y_true)}")
print(f"  Confusion Matrix:")
print(cm)
print(f"{'='*40}")

# Check targets
targets_met = []
if f1 >= 0.68:
    targets_met.append(f"✅ F1 >= 0.68: {f1:.4f}")
else:
    targets_met.append(f"❌ F1 >= 0.68: {f1:.4f}")
    
if recall >= 0.85:
    targets_met.append(f"✅ Recall >= 0.85: {recall:.4f}")
else:
    targets_met.append(f"❌ Recall >= 0.85: {recall:.4f}")
    
if precision >= 0.55:
    targets_met.append(f"✅ Precision >= 0.55: {precision:.4f}")
else:
    targets_met.append(f"❌ Precision >= 0.55: {precision:.4f}")

print("\n🎯 Target Check:")
for t in targets_met:
    print(f"  {t}")

## Step 9: Threshold Optimization

In [ ]:
print("🔍 Optimizing Decision Threshold...\n")

best_f1 = 0
best_threshold = 0.45

for thresh in np.arange(0.30, 0.70, 0.01):
    y_pred_t = (y_prob >= thresh).astype(int)
    f1_t = f1_score(y_true, y_pred_t)
    if f1_t > best_f1:
        best_f1 = f1_t
        best_threshold = thresh

# Final metrics at optimal threshold
y_pred_opt = (y_prob >= best_threshold).astype(int)
f1_opt = f1_score(y_true, y_pred_opt)
precision_opt = precision_score(y_true, y_pred_opt)
recall_opt = recall_score(y_true, y_pred_opt)
accuracy_opt = accuracy_score(y_true, y_pred_opt)
cm_opt = confusion_matrix(y_true, y_pred_opt)

print(f"{'='*40}")
print(f"✨ OPTIMIZED RESULTS (Threshold={best_threshold:.2f})")
print(f"{'='*40}")
print(f"  F1 Score:  {f1_opt:.4f} 🚀")
print(f"  Recall:    {recall_opt:.4f}")
print(f"  Precision: {precision_opt:.4f}")
print(f"  Accuracy:  {accuracy_opt:.4f}")
print(f"  Confusion Matrix:")
print(cm_opt)
print(f"{'='*40}")

## Step 10: Save Final Results

In [ ]:
# Copy final results to Drive
!cp /content/phase8_results.csv "/content/drive/MyDrive/DAIC-WOZ_Datasets/phase8_final_results.csv"
print("✅ Results saved to Google Drive!")

# Summary
print(f"\n{'='*50}")
print(f"🏆 PHASE 8 TRAINING COMPLETE")
print(f"{'='*50}")
print(f"📁 Checkpoints: {OUT_DIR}")
print(f"📊 Results: /content/drive/MyDrive/DAIC-WOZ_Datasets/phase8_final_results.csv")
print(f"\n🎯 Best F1 Score: {best_f1:.4f}")
print(f"{'='*50}")